In [1]:
import pandas as pd

df = df = pd.read_csv("../data/sample_10000.csv")
print(df.shape)
print(df.columns.tolist())

(10000, 56)
['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [2]:
df.head(10)

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1
5,23107.txt,https://www.globalreporting.org,30,www.globalreporting.org,23,0,org,100.0,1.000000,0.079963,...,0,0,1,35,1,11,86,0,14,1
6,23034.txt,https://www.saffronart.com,25,www.saffronart.com,18,0,com,100.0,1.000000,0.522907,...,0,0,1,32,4,14,44,2,17,1
7,696732.txt,https://www.nerdscandy.com,25,www.nerdscandy.com,18,0,com,100.0,1.000000,0.522907,...,0,0,1,24,2,22,36,0,15,1
8,739255.txt,https://www.hyderabadonline.in,29,www.hyderabadonline.in,22,0,in,100.0,1.000000,0.005084,...,0,0,1,71,4,9,40,1,317,1
9,14486.txt,https://www.aap.org,18,www.aap.org,11,0,org,100.0,1.000000,0.079963,...,0,0,1,10,1,12,173,6,65,1


In [3]:
print(df["label"].value_counts())

label
1    6149
0    3851
Name: count, dtype: int64


In [4]:
df["is_phishing"] = (df["label"] == 0).astype(int)

In [5]:
df[["URL", "Domain", "label", "is_phishing"]].head(10)

,URL,Domain,label,is_phishing
0,https://www.southbankmosaics.com,www.southbankmosaics.com,1,0
1,https://www.uni-mainz.de,www.uni-mainz.de,1,0
2,https://www.voicefmradio.co.uk,www.voicefmradio.co.uk,1,0
3,https://www.sfnmjournal.com,www.sfnmjournal.com,1,0
4,https://www.rewildingargentina.org,www.rewildingargentina.org,1,0
5,https://www.globalreporting.org,www.globalreporting.org,1,0
6,https://www.saffronart.com,www.saffronart.com,1,0
7,https://www.nerdscandy.com,www.nerdscandy.com,1,0
8,https://www.hyderabadonline.in,www.hyderabadonline.in,1,0
9,https://www.aap.org,www.aap.org,1,0


### Define function for retreve doman name

In [6]:
from urllib.parse import urlsplit

def extract_hostname(url):
    try:
        if not url.startswith(("http://", "https://")):
            url = "http://" + url

        return urlsplit(url).hostname
    except Exception:
        return None

In [7]:
import tldextract

def get_registered_domain(hostname):
    extracted = tldextract.extract(hostname)

    if extracted.domain and extracted.suffix:
        return f"{extracted.domain}.{extracted.suffix}"

    return hostname

In [8]:
test_url = "https://www.example.com/login/verify?id=123"

print(extract_hostname(test_url))

www.example.com


In [9]:
import dns.resolver

def get_ip_addresses(domain):
    try:
        answers = dns.resolver.resolve(domain, "A")
        return [answer.to_text() for answer in answers]
    except Exception:
        return []

In [10]:
domain = extract_hostname("https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset")

registered_domain = get_registered_domain(domain)

print(domain)
print(registered_domain)

print(get_ip_addresses(domain))

archive.ics.uci.edu
uci.edu
['128.195.10.252']


In [11]:
def get_name_servers(domain):
    try:
        answers = dns.resolver.resolve(domain, "NS")
        return [answer.to_text().rstrip(".") for answer in answers]
    except Exception:
        return []

In [12]:
print(get_name_servers(registered_domain))

['ns5.service.uci.edu', 'ns6.service.uci.edu']


### This function will add two new column for our dataset as ip_addresses and name_servers

In [13]:
def enrich_domain(domain, registered_domain):
    return {
        "ip_addresses": get_ip_addresses(domain),
        "name_servers": get_name_servers(registered_domain)
    }

In [14]:
domain = extract_hostname("https://www.example.com")
registered_domain = get_registered_domain(domain)

result = enrich_domain(domain, registered_domain)

print(result)

{'ip_addresses': ['172.66.147.243', '104.20.23.154'], 'name_servers': ['hera.ns.cloudflare.com', 'elliott.ns.cloudflare.com']}


### For testing we get only 100 records

In [15]:
test_df = df.head(100).copy()

test_df["domain"] = test_df["URL"].apply(extract_hostname)
test_df["registered_domain"] = test_df["domain"].apply(get_registered_domain)

test_df["ip_addresses"] = test_df["domain"].apply(get_ip_addresses)
test_df["name_servers"] = test_df["registered_domain"].apply(get_name_servers)

test_df[
    ["URL", "label", "domain", "ip_addresses", "name_servers"]
].head(20)

KeyboardInterrupt: 

### Testing no of usable rows ( having both ip and name servers )

In [ ]:
has_ip = test_df["ip_addresses"].apply(len) > 0

print("URLs with IP:", has_ip.sum())
print("Percentage:", has_ip.mean() * 100)

In [ ]:
has_ns = test_df["name_servers"].apply(len) > 0

print("URLs with NS:", has_ns.sum())
print("Percentage:", has_ns.mean() * 100)

In [ ]:
has_both = has_ip & has_ns

print("URLs with both:", has_both.sum())
print("Percentage:", has_both.mean() * 100)

### Testing relashionships with ip addresses and name server with phishing

In [ ]:
from collections import Counter

all_ips = []

for ips in test_df["ip_addresses"]:
    all_ips.extend(ips)

ip_counts = Counter(all_ips)

shared_ips = {
    ip: count
    for ip, count in ip_counts.items()
    if count > 1
}

print(shared_ips)

In [ ]:
all_ns = []

for ns_list in test_df["name_servers"]:
    all_ns.extend(ns_list)

ns_counts = Counter(all_ns)

shared_ns = {
    ns: count
    for ns, count in ns_counts.items()
    if count > 1
}

print(shared_ns)

In [ ]:
from collections import defaultdict

ip_urls = defaultdict(list)

for _, row in test_df.iterrows():
    for ip in row["ip_addresses"]:
        ip_urls[ip].append({
            "url": row["URL"],
            "label": row["label"]
        })

In [ ]:
for ip, urls in ip_urls.items():
    if len(urls) > 1:
        print("\nIP:", ip)

        for item in urls:
            label_text = (
                "Phishing"
                if item["label"] == 0
                else "Legitimate"
            )

            print(label_text, ":", item["url"])

In [ ]:
from collections import defaultdict

ns_urls = defaultdict(list)

for _, row in test_df.iterrows():
    for ns in row["name_servers"]:
        ns_urls[ns].append({
            "url": row["URL"],
            "label": row["label"]
        })

In [ ]:
for ns, urls in ns_urls.items():
    if len(urls) > 1:
        print("\nName Server:", ns)

        for item in urls:
            label_text = (
                "Phishing"
                if item["label"] == 0
                else "Legitimate"
            )

            print(label_text, ":", item["url"])

### Testing with whole dataset

In [ ]:
df["domain"] = df["URL"].apply(extract_hostname)
df["registered_domain"] = df["domain"].apply(get_registered_domain)

df["ip_addresses"] = df["domain"].apply(get_ip_addresses)
df["name_servers"] = df["registered_domain"].apply(get_name_servers)

df[
    ["URL", "label", "domain", "ip_addresses", "name_servers"]
].head(20)

#### URLs with IP

In [ ]:
has_ip = df["ip_addresses"].apply(len) > 0

print("URLs with IP:", has_ip.sum())
print("Percentage:", has_ip.mean() * 100)

#### URLs with NS

In [ ]:
has_ns = df["name_servers"].apply(len) > 0

print("URLs with NS:", has_ns.sum())
print("Percentage:", has_ns.mean() * 100)

#### URLs having both

In [ ]:
has_both = has_ip & has_ns

print("URLs with both:", has_both.sum())
print("Percentage:", has_both.mean() * 100)

### Save current data list to a seperate file

In [ ]:
enriched_df = df.copy()
print(enriched_df.shape)
print(enriched_df.columns.tolist())

In [ ]:
enriched_df.to_csv(
    "phiusill_enriched_10000.csv",
    index=False
)

In [ ]:
df.head(10)